# Interlacing null data back into primary datasets

The process of creating the null benchmark was chaotic and required at the end of the day calculating null datasets separate from the primary data. This means we now have to interlace the null results back in with the original data (and label it null).

One thing I did to help make this process a little easier is save the null permuted data for each conversation in a file for that specific conversation--think: if it's CANDOR and the conversation id is 12345, the null data is saved in a file titled `CANDOR-CANDOR-1234.parquet`. This makers interleaving/lacing the data back together a little easier, though it still requires finagling.

The process below is such that we iterate through all the `lme_ready` files, find the appropriate null permuted conversation, and then stitch the data from the null permuted conversation into the original data and resave the original.

> _To ensure that I don't overwrite or lose data, I'm going to have to make sure it's possible to remove all data labeled `null` in the final versions._

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import warnings; warnings.filterwarnings('ignore')

In [2]:
lme_ready_data_path = '../data/ckpts/rosen/lme-ready'
null_data_path = '../data/null-ckpts'

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
lme_ready = [f for f in os.listdir(lme_ready_data_path) if not f.startswith('.')]
null_data = [f for f in os.listdir(null_data_path) if not f.startswith('.')]
len(lme_ready), len(null_data), len(set(['xling-'+parq for parq in lme_ready]).difference(set(null_data)))

(1290, 1290, 27)

In [5]:
set(['xling-'+parq for parq in lme_ready]).difference(set(null_data))

{'xling-yiddish-yiddish-01.parquet',
 'xling-yiddish-yiddish-02.parquet',
 'xling-yiddish-yiddish-03.parquet',
 'xling-yiddish-yiddish-04.parquet',
 'xling-yiddish-yiddish-05.parquet',
 'xling-yiddish-yiddish-06.parquet',
 'xling-yiddish-yiddish-07.parquet',
 'xling-yiddish-yiddish-08.parquet',
 'xling-yiddish-yiddish-09.parquet',
 'xling-yiddish-yiddish-1.parquet',
 'xling-yiddish-yiddish-10.parquet',
 'xling-yiddish-yiddish-11.parquet',
 'xling-yiddish-yiddish-12.parquet',
 'xling-yiddish-yiddish-13.parquet',
 'xling-yiddish-yiddish-14.parquet',
 'xling-yiddish-yiddish-15.parquet',
 'xling-yiddish-yiddish-16.parquet',
 'xling-yiddish-yiddish-2.parquet',
 'xling-yiddish-yiddish-2b.parquet',
 'xling-yiddish-yiddish-3.parquet',
 'xling-yiddish-yiddish-4.parquet',
 'xling-yiddish-yiddish-5.parquet',
 'xling-yiddish-yiddish-5b.parquet',
 'xling-yiddish-yiddish-6.parquet',
 'xling-yiddish-yiddish-7.parquet',
 'xling-yiddish-yiddish-8.parquet',
 'xling-yiddish-yiddish-9.parquet'}

In [6]:
null_yid = [f for f in null_data if 'yiddish' in f]
obs_yid = ['xling-'+f.replace('yiddish-yiddish', 'yiddish-yid-yiddish') for f in lme_ready if 'yiddish' in f]
set(obs_yid).difference(set(null_yid))

set()

In [7]:
# lme_ready[:5], null_data[:5]

## Data processing

In [8]:
for obs_f in tqdm(lme_ready):
    # print(obs_f)
    ######### load observed data
    df = pd.read_parquet(
        os.path.join(lme_ready_data_path, obs_f),
    )
    df['null'] = False

    null_f = 'xling-'+obs_f
    if 'yiddish' in null_f:
        null_f = null_f.replace('yiddish-yiddish', 'yiddish-yid-yiddish')

    df_null = pd.read_parquet(
        os.path.join(null_data_path, null_f),
    )
    df_null['null'] = True
    df_null['x_turn_id'] = df_null['corpus'] + '-' + df_null['conversation_id'] + '-' + df_null['x_turn_id'].astype(str)

    df = pd.concat([df, df_null], ignore_index=True)

    df.to_parquet(
        os.path.join(lme_ready_data_path, obs_f),
        engine='fastparquet',
        compression='snappy'
    )

  0%|          | 0/1290 [00:00<?, ?it/s]

In [ ]:
# lme_ready = grab_all_files(lme_ready_data_path)
# null_data = grab_all_files(null_data_path)

In [ ]:
# search_stats = []
# for obs_f in tqdm(lme_ready):
#     # print(obs_f)
#     ######### load observed data
#     df = pd.read_parquet(obs_f, engine='fastparquet')
#     # add column for null indicator
#     df['null'] = False
#     # create name of likely null data files
#     if 'CANDOR' not in obs_f:
#         df['null_data_file_name'] = null_data_path + '/' + df['corpus'] + '-' + df['conversation_id'] + '.parquet'
#     else:
#         df['null_data_file_name'] = null_data_path + '/CANDOR-' + df['conversation_id'] + '.parquet'
#
#     null_file_names = df['null_data_file_name'].unique()
#
#     ######### load null data
#     # get null files
#     df_null = [fn for fn in null_data if fn in null_file_names]
#
#     search_stats += [
#         {
#             'file': obs_f,
#             'null_found': len(df_null),
#             'observed': len(null_file_names),
#             'equal': sum([fn in null_file_names for fn in df_null]) == len(df_null),
#             'corpus': obs_f.split('/')[-1].split('-')[0]
#         }
#     ]
#
#     # delete now useless column in real data
#     del df['null_data_file_name']
#
#     # load and merge the null files
#     df_null = pd.concat([pd.read_parquet(fn) for fn in df_null],ignore_index=True)
#     df_null['x_turn_id'] = df_null['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)
#     # label as null
#     df_null['null'] = True
#
#     # print(df_null['conversation_id'].isin(df['conversation_id']).mean())
#
#     ######## merge and save data
#     df = pd.concat([df, df_null], ignore_index=True)
#     df.to_parquet(obs_f, engine='fastparquet', compression='snappy')
#
# pd.DataFrame(search_stats).to_csv('/Users/zacharyrosen/Desktop/search_stats.csv', index=False, encoding='utf-8')